# Focus Coach — browser demo (Gradio)

Runs the Focus Coach pipeline (YOLO26n head + phone detection, rules, time smoothing, session report) in your browser, with no install.

**How to use:**
1. Runtime → Change runtime type → **T4 GPU**. If Colab says the GPU limit is reached, choose *Connect without GPU*: it works, but the live tab is slower.
2. Runtime → **Run all**. When the upload button appears in the model cell, choose your `best.pt`.
3. Open the `lhr.life` link printed at the end **in a new tab**.
4. In the Live tab: click the webcam box, allow the camera, then click **Record**. The session starts by itself and the right picture shows what Focus Coach sees. Use your phone for a few seconds, then click **End session** for the report.

The link is temporary and stops working when this Colab session ends. If the live picture is laggy, use the **Video** tab.

**Config.** The only cell you may need to edit: your repo, and where the trained model is.

In [ ]:
REPO_URL = "https://github.com/majallen3690/focus-coach"  # EDIT: the GitHub repo that holds this project
MODEL_PATH = "/content/drive/MyDrive/focus-coach/runs/yolo26n_fpi/weights/best.pt"  # where Phase 2 saved the model on Drive
USE_DRIVE = False  # True: copy best.pt from Google Drive (MODEL_PATH) instead of uploading it

**Runtime check.** Prints the GPU name. A CPU is fine for the Video and Image tabs, but the live tab will be slower.

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. The demo works on a CPU but is slower. For a GPU: Runtime > Change runtime type > T4 GPU.")

**Get the code.** Clones your repo. If it is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left bar → Add new secret → turn on *Notebook access*). The token needs read access to the repo.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/focus-coach")
assert "YOUR-USERNAME" not in REPO_URL, "Edit REPO_URL in the config cell first."

if not REPO.exists():
    url = REPO_URL
    try:
        from google.colab import userdata

        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:  # no GITHUB_TOKEN secret: assume the repo is public
        pass
    done = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
    if done.returncode:  # print git's message without the token that is inside the URL
        raise RuntimeError(done.stderr.replace(url, REPO_URL))
os.chdir(REPO)
print("repo ready at", REPO)

**Get the model.** Copies `best.pt` from Drive, or shows an upload button when `USE_DRIVE = False`. It is saved as `models/best.pt`.

In [ ]:
import shutil

(REPO / "models").mkdir(exist_ok=True)
BEST = REPO / "models" / "best.pt"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    assert Path(MODEL_PATH).exists(), f"No model at {MODEL_PATH}. Fix MODEL_PATH in the config cell, or set USE_DRIVE = False."
    shutil.copy(MODEL_PATH, BEST)
else:
    from google.colab import files

    print("Choose your best.pt file:")
    uploaded = files.upload()
    assert uploaded, "No file was uploaded."
    shutil.move(next(iter(uploaded)), BEST)
print("model ready:", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")

**Install** the two packages Colab does not have (Ultralytics and Gradio, pinned). Colab already has numpy, pandas, OpenCV, matplotlib and torch, so they are left alone on purpose: reinstalling them breaks Colab's own packages. Red lines about other packages are harmless; if Colab asks to restart the runtime, restart and Run all again.

In [ ]:
!pip install -q ultralytics==8.4.190 gradio==6.31.0

**Smoke test.** Runs the model on one example photo and checks the answer. Stops here with a clear message if something is wrong, before the demo starts.

In [ ]:
import sys

sys.path.insert(0, str(REPO))
from app.pipeline import FramePipeline
from ml import rules
import cv2

cfg = rules.load_config()
pipeline = FramePipeline(cfg, weights=BEST)
example = REPO / "demo/examples/images/03_on_phone.jpg"
result = pipeline.process(cv2.imread(str(example)), "off")
for d in result.dets:
    print(f"  {d['cls']:5s} {d['conf']:.2f}")
assert result.state == rules.ON_PHONE, f"Smoke test failed: expected ON_PHONE for {example.name}, got {result.state}. Is best.pt the right model?"
print("Smoke test passed: the model sees the phone and the rules say", result.state)

**Launch.** Starts the demo with no outside tunnel:
- **Video and Image tabs:** use them right here, inside the notebook output below the cell.
- **Live webcam:** the cell prints a link. Open it **in a new tab** (a page inside the notebook cannot use the camera), allow the camera, then click **Record**.
  The link goes through Colab itself, so it only opens in a browser signed in to the same Google account as this notebook.

This cell can be run again; it restarts the demo.

In [ ]:
import importlib
import subprocess

from google.colab.output import eval_js
from IPython.display import HTML, display

subprocess.run(["git", "-C", str(REPO), "pull", "-q"])  # get the latest demo code
import demo.gradio_app as gradio_app
importlib.reload(gradio_app)

for name in ("demo", "app"):  # running this cell again: stop the previous demo first
    old = globals().get(name)
    if hasattr(old, "close"):
        old.close()

live_url = eval_js("google.colab.kernel.proxyPort(7860)")  # Colab's own address for the demo: no outside tunnel needed
display(HTML(f'<h2>Live webcam: <a href="{live_url}" target="_blank">open this in a new tab</a></h2>'
             "<p>Allow the camera, then click <b>Record</b>. Click <b>End session</b> for the report. "
             "Video and Image upload work in the box below. Frames are processed in this session and not stored.</p>"))
app = gradio_app.build_demo(model_path=BEST)
app.queue().launch(server_port=7860, share=False, inline=True, quiet=True, prevent_thread_lock=True)
app.block_thread()